In [32]:
# SolarSense Model 4 - Integration Test Notebook
# Fixed version - handles None values gracefully

import requests
import json
import pandas as pd
from datetime import datetime

# Configuration
BASE_URL = "http://127.0.0.1:5000"  # Local Model 4
API_KEY = "QfBmvo3htYQPSJWYYLcChxXvZonTuC2U"

print("✅ Setup complete!")
print(f"🔗 Testing API at: {BASE_URL}")
print()     

✅ Setup complete!
🔗 Testing API at: http://127.0.0.1:5000



In [33]:
# ============================================
# TEST 1: Health Check
# ============================================
print("📋 TEST 1: Health Check")
print("-" * 50)

try:
    response = requests.get(f"{BASE_URL}/health")
    health_data = response.json()
    print(f"Status: {health_data.get('status', 'Unknown')}")
    print(f"Service: {health_data.get('service', 'Unknown')}")
    print(f"Version: {health_data.get('version', 'Unknown')}")
except Exception as e:
    print(f"❌ Health check failed: {e}")

print()

📋 TEST 1: Health Check
--------------------------------------------------


Status: healthy
Service: Model 4 Orchestration (Enhanced)
Version: 1.1.0



In [34]:
# ============================================
# TEST 2: Multiple Business Scenarios
# ============================================
print("📋 TEST 2: Multiple Business Scenarios")
print("-" * 50)

test_businesses = [
    {
        "business_name": "Joe's Coffee Shop",
        "business_type": "Cafe",
        "location": "Nairobi, Kenya",
        "latitude": -1.2921,
        "longitude": 36.8219,
        "equipment_list": [
            {"equipment_type": "refrigerator", "quantity": 2, "hours_per_day": 24},
            {"equipment_type": "espresso_machine", "quantity": 1, "hours_per_day": 10},
            {"equipment_type": "lights", "quantity": 15, "hours_per_day": 12},
            {"equipment_type": "air_conditioner", "quantity": 2, "hours_per_day": 8}
        ],
        "tariff": 0.15
    },
    {
        "business_name": "TechHub Office",
        "business_type": "Office",
        "location": "Nairobi, Kenya",
        "latitude": -1.2921,
        "longitude": 36.8219,
        "equipment_list": [
            {"equipment_type": "computer", "quantity": 20, "hours_per_day": 9},
            {"equipment_type": "printer", "quantity": 3, "hours_per_day": 4},
            {"equipment_type": "air_conditioner", "quantity": 3, "hours_per_day": 9},
            {"equipment_type": "lights", "quantity": 30, "hours_per_day": 10}
        ],
        "tariff": 0.18
    },
    {
        "business_name": "Coastal Restaurant",
        "business_type": "Restaurant",
        "location": "Mombasa, Kenya",
        "latitude": -4.0435,
        "longitude": 39.6682,
        "equipment_list": [
            {"equipment_type": "refrigerator", "quantity": 3, "hours_per_day": 24},
            {"equipment_type": "oven", "quantity": 2, "hours_per_day": 8},
            {"equipment_type": "lights", "quantity": 25, "hours_per_day": 14},
            {"equipment_type": "air_conditioner", "quantity": 4, "hours_per_day": 12}
        ],
        "tariff": 0.16
    },
    {
        "business_name": "FitLife Gym",
        "business_type": "Gym",
        "location": "Nairobi, Kenya",
        "latitude": -1.2921,
        "longitude": 36.8219,
        "equipment_list": [
            {"equipment_type": "treadmill", "quantity": 10, "hours_per_day": 12},
            {"equipment_type": "lights", "quantity": 40, "hours_per_day": 14},
            {"equipment_type": "air_conditioner", "quantity": 5, "hours_per_day": 14},
            {"equipment_type": "water_cooler", "quantity": 3, "hours_per_day": 14}
        ],
        "tariff": 0.17
    },
    {
        "business_name": "Mini Mart",
        "business_type": "Retail",
        "location": "Nakuru, Kenya",
        "latitude": -0.3031,
        "longitude": 36.0800,
        "equipment_list": [
            {"equipment_type": "refrigerator", "quantity": 2, "hours_per_day": 24},
            {"equipment_type": "lights", "quantity": 20, "hours_per_day": 12},
            {"equipment_type": "fan", "quantity": 4, "hours_per_day": 10}
        ],
        "tariff": 0.14
    }
]

results = []
headers = {"X-API-KEY": API_KEY, "Content-Type": "application/json"}

for business in test_businesses:
    try:
        response = requests.post(
            f"{BASE_URL}/api/v1/analyze",
            headers=headers,
            json=business,
            timeout=30
        )
        
        if response.status_code == 200:
            result = response.json()
            
            # Safe extraction with defaults for None values
            consumption = result.get('estimated_consumption_kwh')
            generation = result.get('predicted_generation_kwh')
            panel_area = result.get('recommended_panel_area_m2')
            roi = result.get('roi_percentage')
            savings = result.get('monthly_savings')
            system_cost = result.get('system_cost')
            viability_score = result.get('viability_score')
            
            results.append({
                'Business': result.get('business_name', 'Unknown'),
                'Type': result.get('business_type', 'Unknown'),
                'Location': result.get('location', 'Unknown'),
                'Consumption (kWh/mo)': f"{consumption:.0f}" if consumption is not None else 'N/A',
                'Generation (kWh/mo)': f"{generation:.0f}" if generation is not None else 'N/A',
                'Panel Area (m²)': f"{panel_area:.1f}" if panel_area is not None else 'N/A',
                'System Cost ($)': f"{system_cost:,.0f}" if system_cost is not None else 'N/A',
                'ROI (%)': f"{roi:.1f}" if roi is not None else 'N/A',
                'Monthly Savings ($)': f"{savings:.2f}" if savings is not None else 'N/A',
                'Recommendation': result.get('recommendation', 'Unknown'),
                'Viability Score': f"{viability_score:.1f}" if viability_score is not None else 'N/A',
                'Status': result.get('status', 'Unknown')
            })
            print(f"✅ {business['business_name']}: {result.get('recommendation', 'Unknown')} viability")
        else:
            print(f"❌ {business['business_name']}: Error {response.status_code}")
            error_msg = 'N/A'
            try:
                error_data = response.json()
                error_msg = error_data.get('message', 'Unknown error')
            except:
                error_msg = response.text[:100] if response.text else 'No error message'
            
            results.append({
                'Business': business['business_name'],
                'Type': business['business_type'],
                'Location': business['location'],
                'Consumption (kWh/mo)': 'N/A',
                'Generation (kWh/mo)': 'N/A',
                'Panel Area (m²)': 'N/A',
                'System Cost ($)': 'N/A',
                'ROI (%)': 'N/A',
                'Monthly Savings ($)': 'N/A',
                'Recommendation': 'Error',
                'Viability Score': 'N/A',
                'Status': f'Error: {response.status_code} - {error_msg}'
            })
    except Exception as e:
        print(f"❌ {business['business_name']}: Exception - {str(e)}")
        results.append({
            'Business': business['business_name'],
            'Type': business['business_type'],
            'Location': business['location'],
            'Consumption (kWh/mo)': 'N/A',
            'Generation (kWh/mo)': 'N/A',
            'Panel Area (m²)': 'N/A',
            'System Cost ($)': 'N/A',
            'ROI (%)': 'N/A',
            'Monthly Savings ($)': 'N/A',
            'Recommendation': 'Exception',
            'Viability Score': 'N/A',
            'Status': f'Exception: {str(e)[:100]}'
        })

print()
print("=" * 80)
print("📊 RESULTS SUMMARY")
print("=" * 80)

# Display results as table
if results:
    results_df = pd.DataFrame(results)
    print(results_df.to_string(index=False))
    print()
    print(f"📈 Analyzed {len(results)} businesses")
    print(f"   ✅ Successful: {sum(1 for r in results if r['Status'] not in ['Error', 'Exception'] and 'Error' not in str(r['Status']))}")
    print(f"   ❌ Failed: {sum(1 for r in results if 'Error' in str(r['Status']) or 'Exception' in str(r['Status']))}")
    print()
    print(f"   🟢 High viability: {sum(1 for r in results if r['Recommendation'] == 'High')}")
    print(f"   🟡 Medium viability: {sum(1 for r in results if r['Recommendation'] == 'Medium')}")
    print(f"   🔴 Low viability: {sum(1 for r in results if r['Recommendation'] == 'Low')}")
else:
    print("❌ No results to display")

print()
print("=" * 80)
print("✅ All tests complete!")
print("=" * 80)

📋 TEST 2: Multiple Business Scenarios
--------------------------------------------------


✅ Joe's Coffee Shop: Low viability
✅ TechHub Office: Low viability
✅ Coastal Restaurant: Low viability
✅ FitLife Gym: Low viability
✅ Mini Mart: Low viability

📊 RESULTS SUMMARY
          Business       Type       Location Consumption (kWh/mo) Generation (kWh/mo) Panel Area (m²) System Cost ($) ROI (%) Monthly Savings ($) Recommendation Viability Score     Status
 Joe's Coffee Shop       Cafe Nairobi, Kenya                  221                 199            24.5          14,707     2.4                 N/A            Low            28.2 NOT VIABLE
    TechHub Office     Office Nairobi, Kenya                  233                 209            25.8          15,499     2.9                 N/A            Low            28.5 NOT VIABLE
Coastal Restaurant Restaurant Mombasa, Kenya                  221                 199            16.0           9,578     4.0                 N/A            Low            29.0 NOT VIABLE
       FitLife Gym        Gym Nairobi, Kenya                  221     